# Class-conditional image generation and editing using MaskGIT.

This notebook is an official Colab notebook using pretrained [MaskGIT](https://arxiv.org/pdf/2202.04200.pdf) models for class-conditional image generation.

After connecting to a runtime, get started by following these instructions:

1. Make sure you've selected a GPU accelerator (Runtime > Change runtime type > Hardware accelerator > GPU).
2.Click the **Play** button to the left of the code cell, or use the keyboard shortcut "Command/Ctrl+Enter" to generate.

Install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install jax flax
%pip install numpy tensorflow matplotlib ml_collections
%pip install scikit-commpy

Clone the repository


In [ ]:
# 刪除舊資料夾
!rm -rf /content/maskgit
# 回到根目錄以防萬一
%cd /content/
!git clone -b feat/jax-compat https://github.com/zhengpohung/maskgit
%cd maskgit
%ls

Set up a couple of imports before we dive in.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
if not hasattr(jnp, "DeviceArray"):
    jnp.DeviceArray = jax.Array
import os
import itertools
from timeit import default_timer as timer

import maskgit
from maskgit.utils import visualize_images, read_image_from_url, restore_from_path, draw_image_with_bbox, Bbox
from maskgit.inference import ImageNet_class_conditional_generator
from communication_sim_ARQ import simulate_transmission_analytical
import tensorflow_datasets as tfds
import tensorflow as tf

Download the pretrained models, including

1. two tokenization models, both of which map an image of size $H \times W$ into a latent code of size $H/16 \times W/16$.
2. two masked visual token modeling (MVTM) models, one for $256 \times 256$ and one for $512 \times 512$.


In [ ]:
!mkdir -p checkpoints/

models_to_download = itertools.product(
    *[ ["maskgit", "tokenizer"],   [256, 512] ])

for (type_, resolution) in models_to_download:
  canonical_path = ImageNet_class_conditional_generator.checkpoint_canonical_path(type_, resolution)
  if os.path.isfile(canonical_path):
    print(f"Checkpoint for {resolution} {type_} already exists, not downloading again")
  else:
    source_url = f'https://storage.googleapis.com/maskgit-public/checkpoints/{type_}_imagenet{resolution}_checkpoint'
    !wget {source_url} -O {canonical_path}

### Initialization & resoultion defaults
Instantiate maskgit generators for both resolutions using the pre-trained checkpoints we just downloaded. By default, the examples for generation use the 256 model, while the image editing examples use the 512 model for demo purposes.

### Run Mode

Let's also choose a **run_mode**, which can be either 'normal' or 'pmap'. By default, 'normal' is enabled.

The 'pmap' mode uses [jax.pmap] under the hood, which offers a substantial  speedup for accelerators that can handle the higher memory
requirement, e.g. TPUs, or GPUs such as V100.

Running 'pmap' mode with a GPU (or CPU) with a smaller memory may lead to OOM crashes. That said,

**if your hardware allows, 'pmap' mode is strongly recommended. Each image can typically be generated in < 1s in 'pmap' mode on a TPU**.

In [ ]:
generator_256 = ImageNet_class_conditional_generator(image_size=256)
generator_512 = ImageNet_class_conditional_generator(image_size=512)
arbitrary_seed = 42
rng = jax.random.PRNGKey(arbitrary_seed)

run_mode = 'normal'  #@param ['normal', 'pmap']

p_generate_256_samples = generator_256.p_generate_samples()
p_edit_512_samples = generator_512.p_edit_samples()

# Class-conditional Image Synthesis

Choose the ImageNet **label**, which determines what type of object to synthesize.

In [ ]:
import os

# --- 路徑設定 ---
# 根目錄指向您在雲端硬碟中的資料夾
drive_root = '/content/drive/MyDrive/MaskGIT'

# JSON 檔案與圖片資料夾的完整路徑
json_path = os.path.join(drive_root, 'class_to_folder.json')
image_folder_root = os.path.join(drive_root, 'ImageNet100')

# 檢查路徑是否存在
if os.path.exists(drive_root):
    print(f"成功連接到雲端硬碟資料夾: {drive_root}")
    if not os.path.exists(json_path):
        print(f"⚠️ 警告: 找不到 JSON 檔案於 {json_path}")
    if not os.path.exists(image_folder_root):
        print(f"⚠️ 警告: 找不到 ImageNet100 資料夾於 {image_folder_root}")
else:
    print(f"❌ 錯誤: 找不到您的資料夾 {drive_root}，請確認路徑是否正確。")


In [ ]:
import json
from PIL import Image
import os
import random
import math
import gc

# --- 路徑設定 ---
# 【修改點】: 將儲存根目錄指回您的 Google 雲端硬碟
drive_root = '/content/drive/MyDrive/MaskGIT'
results_save_root = os.path.join(drive_root, 'simulation_results')
os.makedirs(results_save_root, exist_ok=True)
print(f"所有重建圖片將儲存於: {results_save_root}")

# --- 實驗參數 ---
image_size = 256
snr_values = [6, 8, 10, 12, 14, 16, 18]
model_mask_id = generator_256.maskgit_cf.transformer.mask_token_id
#model_mask_id = 0

# --- 讀取類別對應檔並抽樣 ---
try:
    with open(json_path, 'r', encoding='utf-8') as f:
        class_mapping = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"錯誤：找不到 JSON 檔案於 '{json_path}'。請確認路徑是否正確。")

all_image_paths = []
for label_str, folder_name in class_mapping.items():
    class_folder_path = os.path.join(image_folder_root, folder_name)
    if os.path.isdir(class_folder_path):
        for image_name in os.listdir(class_folder_path):
            if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_image_paths.append((os.path.join(class_folder_path, image_name), int(label_str), image_name))

print(f"共找到 {len(all_image_paths)} 張圖片。")
random.seed(555)
random.shuffle(all_image_paths)
num_to_process = math.ceil(len(all_image_paths) * 0.1)
image_subset = all_image_paths[:num_to_process]
print(f"將處理 1/10 的資料，共 {len(image_subset)} 張圖片。")

# --- 主迴圈：高效批次處理 + 可續跑功能 ---
rng, sample_rng = jax.random.split(rng)
start_timer = timer()
total_image_count = 0
skipped_count = 0

for image_path, source_label, original_filename in image_subset:
    # --- !!【新增的可續跑邏輯】!! ---
    # 檢查這張圖的其中一個最終檔案是否存在於雲端硬碟
    # 我們選擇檢查 SNR=18 的 CMI 版本圖片
    last_snr = snr_values[-1]
    check_filename = f"{source_label}_{os.path.splitext(original_filename)[0]}.png"
    check_path = os.path.join(results_save_root, f"SNR_{last_snr}", "with_cmi", check_filename)

    if os.path.exists(check_path):
        if total_image_count % 50 == 0: # 每隔一段時間提示一次，避免洗版
            print(f"  - 第 {total_image_count + 1} 張圖片 '{original_filename}' 已處理過，跳過。")
        skipped_count += 1
        total_image_count += 1
        continue
    # -----------------------------------------------

    print(f"  - 正在處理第 {total_image_count + 1} / {len(image_subset)} 張圖片: {original_filename}")

    try:
        img = Image.open(image_path).convert('RGB').resize((image_size, image_size))
        source_image_np = np.array(img)
        source_image_ex = np.expand_dims(source_image_np.astype(np.float32) / 255.0, axis=0)
    except Exception as e:
        print(f"讀取圖片 {image_path} 失敗: {e}")
        total_image_count += 1
        continue

    # 1. 編碼一次，得到 perfect_tokens
    _, result_dict = generator_256.tokenizer_model.apply(
        {'params': generator_256.tokenizer_variables['params']},
        {"image": source_image_ex},
        method=generator_256.tokenizer_model.encode
    )
    perfect_tokens = result_dict['encoding_indices']

    # 2. 準備批次
    batch_input_wc, batch_input_woc = [], []
    codebook_size = generator_256.maskgit_cf.vqvae.codebook_size
    for snr_db in snr_values:
        tokens_after_channel = simulate_transmission_analytical(perfect_tokens, snr_db, generator_256)
        label_token_wc = np.array([[source_label + codebook_size]])
        batch_input_wc.append(np.concatenate([label_token_wc, tokens_after_channel], axis=1))
        label_token_woc = np.array([[model_mask_id]])
        batch_input_woc.append(np.concatenate([label_token_woc, tokens_after_channel], axis=1))
    final_batch_wc = np.concatenate(batch_input_wc, axis=0)
    final_batch_woc = np.concatenate(batch_input_woc, axis=0)

    # 3. 模型推理
    results_with_cmi = generator_256.generate_samples(input_tokens=final_batch_wc, rng=sample_rng)
    results_without_cmi = generator_256.generate_samples(input_tokens=final_batch_woc, rng=sample_rng)

    # 4. 儲存結果到【雲端硬碟】
    for i, snr_db in enumerate(snr_values):
        snr_folder = os.path.join(results_save_root, f"SNR_{snr_db}")
        os.makedirs(os.path.join(snr_folder, "original"), exist_ok=True)
        os.makedirs(os.path.join(snr_folder, "with_cmi"), exist_ok=True)
        os.makedirs(os.path.join(snr_folder, "without_cmi"), exist_ok=True)
        base_filename = f"{source_label}_{os.path.splitext(original_filename)[0]}.png"
        original_save_path = os.path.join(snr_folder, "original", base_filename)
        if not os.path.exists(original_save_path):
            Image.fromarray(source_image_np).save(original_save_path)
        reconstructed_wc = (np.clip(results_with_cmi[i], 0, 1) * 255).astype(np.uint8)
        reconstructed_woc = (np.clip(results_without_cmi[i], 0, 1) * 255).astype(np.uint8)
        wc_save_path = os.path.join(snr_folder, "with_cmi", base_filename)
        woc_save_path = os.path.join(snr_folder, "without_cmi", base_filename)
        Image.fromarray(reconstructed_wc).save(wc_save_path)
        Image.fromarray(reconstructed_woc).save(woc_save_path)

    # 手動釋放記憶體
    del img, source_image_np, source_image_ex, result_dict, perfect_tokens, batch_input_wc, batch_input_woc, final_batch_wc, final_batch_woc, results_with_cmi, results_without_cmi, reconstructed_wc, reconstructed_woc
    gc.collect()
    total_image_count += 1

end_timer = timer()
print(f"\n✅ 模擬完成！")
print(f"總共掃描了 {total_image_count} 張圖片。")
print(f"其中 {skipped_count} 張圖片已被處理過並跳過。")
print(f"本次執行實際處理了 {total_image_count - skipped_count} 張新圖片。")
print(f"總耗時: {(end_timer - start_timer) / 60:.2f} 分鐘。")

# Class-conditional Image Editing

In [ ]:
# --- 全新的 Metric Caclulation Cell (從檔案讀取) ---

# --- 安裝與匯入函式庫 ---
%pip install clip-openai lpips scikit-image pandas -q
import torch
from PIL import Image
import clip
import lpips as lpips_lib
from skimage.metrics import peak_signal_noise_ratio as psnr
import numpy as np
import os
import pandas as pd

# 假設此檔案與 notebook 在同一路徑下
from communication_sim_ARQ import calculate_analytical_per

# --- 初始化指標計算模型 ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
lpips_fn = lpips_lib.LPIPS(net='alex').to(device)

# --- 指標計算輔助函式 (不變) ---
def calculate_metrics(original_img, reconstructed_img):
    original_img = np.asarray(original_img)
    reconstructed_img = np.asarray(reconstructed_img)
    psnr_val = psnr(original_img, reconstructed_img, data_range=255)
    def to_tensor(img):
        img_pil = Image.fromarray(img)
        return (torch.from_numpy(np.array(img_pil)).permute(2, 0, 1).unsqueeze(0).to(device) / 127.5) - 1.0
    original_tensor = to_tensor(original_img)
    reconstructed_tensor = to_tensor(reconstructed_img)
    with torch.no_grad():
        lpips_val = lpips_fn(original_tensor, reconstructed_tensor).item()
        image1 = clip_preprocess(Image.fromarray(original_img)).unsqueeze(0).to(device)
        image2 = clip_preprocess(Image.fromarray(reconstructed_img)).unsqueeze(0).to(device)
        image_features1 = clip_model.encode_image(image1)
        image_features2 = clip_model.encode_image(image2)
        cos_sim = torch.nn.functional.cosine_similarity(image_features1, image_features2)
        clip_score = cos_sim.item()
    return psnr_val, lpips_val, clip_score

def calculate_tce_tokcom():
    h, w, N, Q = 256, 256, 256, 1024
    T = 1
    return (h * w) / (T * N * np.log2(Q))

# --- 主邏輯：遍歷已儲存的檔案並計算指標 ---
metrics_results = {
    "snr": [], "per": [], "tce_tokcom": [],
    "psnr_tokcom_with_cmi": [], "lpips_tokcom_with_cmi": [], "clip_tokcom_with_cmi": [],
    "psnr_tokcom_without_cmi": [], "lpips_tokcom_without_cmi": [], "clip_tokcom_without_cmi": [],
}

tce_value = calculate_tce_tokcom()
snr_values = [6, 8, 10, 12, 14, 16, 18]
# results_save_root 需與上一個 cell 相同
results_save_root = '/content/drive/MyDrive/MaskGIT/simulation_results'
# --- 【修改點】: 使用與模擬時相同的 per_map 字典 ---
per_map = {
    6: 0.41, 8: 0.29, 10: 0.19, 12: 0.13,
    14: 0.08, 16: 0.05, 18: 0.03
}

print("開始從已儲存的圖片計算指標...")
for snr in snr_values:
    per = per_map.get(snr, 0)
    metrics_results["snr"].append(snr)
    metrics_results["per"].append(per)
    metrics_results["tce_tokcom"].append(tce_value)

    print(f"  - 正在計算 SNR = {snr} dB 的指標...")

    snr_folder = os.path.join(results_save_root, f"SNR_{snr}")
    original_folder = os.path.join(snr_folder, "original")
    wc_folder = os.path.join(snr_folder, "with_cmi")
    woc_folder = os.path.join(snr_folder, "without_cmi")

    if not os.path.isdir(wc_folder):
        print(f"  - 警告: 找不到 {wc_folder}，跳過此 SNR。")
        # 填入空值或 NaN 以維持 DataFrame 結構
        for key in metrics_results:
            if key not in ["snr", "per", "tce_tokcom"]:
                metrics_results[key].append(np.nan)
        continue

    psnr_wc, lpips_wc, clip_wc = [], [], []
    psnr_woc, lpips_woc, clip_woc = [], [], []
    processed_count = 0
    # 以 with_cmi 資料夾中的檔案為基準進行遍歷
    for filename in os.listdir(wc_folder):
        orig_path = os.path.join(original_folder, filename)
        wc_path = os.path.join(wc_folder, filename)
        woc_path = os.path.join(woc_folder, filename)

        if os.path.exists(orig_path) and os.path.exists(woc_path):
            try:
                orig_img = Image.open(orig_path)

                # 計算 with CMI
                wc_img = Image.open(wc_path)
                p, l, c = calculate_metrics(orig_img, wc_img)
                psnr_wc.append(p); lpips_wc.append(l); clip_wc.append(c)

                # 計算 without CMI
                woc_img = Image.open(woc_path)
                p, l, c = calculate_metrics(orig_img, woc_img)
                psnr_woc.append(p); lpips_woc.append(l); clip_woc.append(c)
                processed_count += 1
            except Exception as e:
                print(f"處理檔案 {filename} 時出錯: {e}")

    # 計算平均值並儲存
    metrics_results["psnr_tokcom_with_cmi"].append(np.mean(psnr_wc) if psnr_wc else 0)
    metrics_results["lpips_tokcom_with_cmi"].append(np.mean(lpips_wc) if lpips_wc else 0)
    metrics_results["clip_tokcom_with_cmi"].append(np.mean(clip_wc) if clip_wc else 0)

    metrics_results["psnr_tokcom_without_cmi"].append(np.mean(psnr_woc) if psnr_woc else 0)
    metrics_results["lpips_tokcom_without_cmi"].append(np.mean(lpips_woc) if lpips_woc else 0)
    metrics_results["clip_tokcom_without_cmi"].append(np.mean(clip_woc) if clip_woc else 0)
    print(f"    -> 使用了 {processed_count} 張圖片計算平均指標。")

print("\n指標計算完成！")
df_metrics = pd.DataFrame(metrics_results)
display(df_metrics)

In [ ]:
# (替換後的繪圖 Cell)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import random

# --- 步驟 A: 修正 "Perfect Transmission" 基準線的計算方式 ---

# 我們從已儲存的檔案中，隨機讀取幾張原始圖片來計算理論最佳值
baseline_psnr, baseline_lpips, baseline_clip = [], [], []

# 從任何一個 SNR 資料夾中取得原始圖片的路徑列表
snr_for_baseline = snr_values[0] # 用第一個 SNR 值當作代表
original_folder_path = os.path.join(results_save_root, f"SNR_{snr_for_baseline}", "original")

if os.path.isdir(original_folder_path):
    original_image_files = [os.path.join(original_folder_path, f) for f in os.listdir(original_folder_path)]
    # 隨機選取最多 10 張圖片來計算平均基準線，以節省時間
    num_baseline_images = min(10, len(original_image_files))
    baseline_image_paths = random.sample(original_image_files, num_baseline_images)

    print(f"正在使用 {num_baseline_images} 張圖片計算 'Perfect Transmission' 基準線...")

    for orig_img_path in baseline_image_paths:
        try:
            orig_img = Image.open(orig_img_path)
            orig_img_np = np.array(orig_img)

            # 1. 準備輸入
            source_image_batch = np.expand_dims(orig_img_np.astype(np.float32) / 255.0, axis=0)

            # 2. VQGAN 編碼
            _, result_dict = generator_256.tokenizer_model.apply(
                {'params': generator_256.tokenizer_variables['params']},
                {"image": source_image_batch},
                method=generator_256.tokenizer_model.encode
            )
            perfect_tokens = result_dict['encoding_indices']

            # 3. VQGAN 解碼
            reconstructed_image_perfect = generator_256.tokenizer_model.apply(
                generator_256.tokenizer_variables,
                perfect_tokens,
                method=generator_256.tokenizer_model.decode_from_indices
            )
            reconstructed_image_perfect_np = np.array(reconstructed_image_perfect[0] * 255, dtype=np.uint8)

            # 4. 計算指標
            p, l, c = calculate_metrics(orig_img_np, reconstructed_image_perfect_np)
            baseline_psnr.append(p)
            baseline_lpips.append(l)
            baseline_clip.append(c)
        except Exception as e:
            print(f"計算基準線時讀取檔案 {orig_img_path} 失敗: {e}")

# 5. 計算平均值作為最終的基準線
if baseline_psnr:
    perfect_psnr_baseline = np.mean(baseline_psnr)
    perfect_lpips_baseline = np.mean(baseline_lpips)
    perfect_clip_baseline = np.mean(baseline_clip)
    print(f"計算出的 'Perfect Transmission' 基準線:")
    print(f"PSNR: {perfect_psnr_baseline:.2f}, LPIPS: {perfect_lpips_baseline:.4f}, CLIP: {perfect_clip_baseline:.4f}")
else:
    print("警告：基準線計算失敗或找不到原始圖片，將使用預設值 0。")
    perfect_psnr_baseline, perfect_lpips_baseline, perfect_clip_baseline = 0, 0, 0


# --- 步驟 B: 繪製所有結果圖 (使用 metrics_results) ---
fig, axs = plt.subplots(4, 1, figsize=(12, 20), sharex=True)
plt.rcParams.update({'font.size': 12})

snr_ticks = metrics_results["snr"]
per_ticks = metrics_results["per"]

# 1. 繪製 TCE
ax1 = axs[0]
ax1.plot(snr_ticks, metrics_results["tce_tokcom"], 's-', color='purple', label='TokCom (w/ & w/o CMI)')
ax1.set_ylabel('TCE (↑)')
ax1.set_title('TokCom Performance Metrics vs. SNR and PER', pad=30)
ax1.grid(True, linestyle='--')
ax1.legend(loc='lower right')
ax1.set_ylim(bottom=0)

ax1b = ax1.twiny()
ax1b.set_xlabel('Packet Error Rate (PER)')
ax1b.set_xlim(ax1.get_xlim())
ax1b.set_xticks(snr_ticks)
ax1b.set_xticklabels([f'{p:.2f}' for p in per_ticks])

# 2. 繪製 CLIP
axs[1].plot(snr_ticks, metrics_results["clip_tokcom_with_cmi"], 's-', color='orange', label='TokCom w/ CMI')
axs[1].plot(snr_ticks, metrics_results["clip_tokcom_without_cmi"], 'o--', color='green', label='TokCom w/o CMI')
axs[1].axhline(y=perfect_clip_baseline, color='blue', linestyle='--', label='Perfect Transmission')
axs[1].set_ylabel('CLIP Score (↑)')
axs[1].grid(True, linestyle='--')
axs[1].legend(loc='lower right')

# 3. 繪製 LPIPS
axs[2].plot(snr_ticks, metrics_results["lpips_tokcom_with_cmi"], 's-', color='orange', label='TokCom w/ CMI')
axs[2].plot(snr_ticks, metrics_results["lpips_tokcom_without_cmi"], 'o--', color='green', label='TokCom w/o CMI')
axs[2].axhline(y=perfect_lpips_baseline, color='blue', linestyle='--', label='Perfect Transmission')
axs[2].set_ylabel('LPIPS (↓)')
axs[2].grid(True, linestyle='--')
axs[2].legend(loc='upper right')

# 4. 繪製 PSNR
axs[3].plot(snr_ticks, metrics_results["psnr_tokcom_with_cmi"], 's-', color='orange', label='TokCom w/ CMI')
axs[3].plot(snr_ticks, metrics_results["psnr_tokcom_without_cmi"], 'o--', color='green', label='TokCom w/o CMI')
axs[3].axhline(y=perfect_psnr_baseline, color='blue', linestyle='--', label='Perfect Transmission')
axs[3].set_xlabel('SNR (dB)')
axs[3].set_ylabel('PSNR (↑)')
axs[3].grid(True, linestyle='--')
axs[3].legend(loc='lower right')

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# --- 步驟 C: 修正後的視覺化 (從檔案讀取範例圖片) ---
def show_comparison_images(snr, num_images=4):
    print(f"\n--- 比較 SNR={snr}dB 的重建結果 (隨機 {num_images} 張範例) ---")

    snr_folder = os.path.join(results_save_root, f"SNR_{snr}")
    original_folder = os.path.join(snr_folder, "original")
    wc_folder = os.path.join(snr_folder, "with_cmi")
    woc_folder = os.path.join(snr_folder, "without_cmi")

    if not os.path.isdir(original_folder):
        print(f"找不到 {original_folder}，無法顯示範例圖片。")
        return

    # 從 original 資料夾隨機選取檔名
    try:
        sample_filenames = random.sample(os.listdir(original_folder), num_images)
    except ValueError: # 如果檔案少於 num_images
        sample_filenames = os.listdir(original_folder)
        print(f"找到的檔案少於 {num_images} 張，將全部顯示。")

    if not sample_filenames:
        print("找不到可供顯示的圖片。")
        return

    try:
        # 根據檔名讀取三種版本的圖片
        original_imgs = np.stack([np.array(Image.open(os.path.join(original_folder, f))) for f in sample_filenames], axis=0)
        wc_imgs = np.stack([np.array(Image.open(os.path.join(wc_folder, f))) for f in sample_filenames], axis=0)
        woc_imgs = np.stack([np.array(Image.open(os.path.join(woc_folder, f))) for f in sample_filenames], axis=0)

        # 視覺化
        visualize_images(original_imgs / 255.0, title=f'Original Images')
        visualize_images(wc_imgs / 255.0, title=f'Reconstructed with CMI @ SNR={snr}dB')
        visualize_images(woc_imgs / 255.0, title=f'Reconstructed without CMI @ SNR={snr}dB')
    except FileNotFoundError as e:
        print(f"顯示圖片失敗，找不到對應的檔案: {e}")
    except Exception as e:
        print(f"顯示圖片時發生錯誤: {e}")

# 顯示高 SNR 和低 SNR 的比較
show_comparison_images(18)
show_comparison_images(6)